# MentalDistress Phase 11 — Mechanism, Reliability, and Lexical Robustness (GPU)

Phase 11 explains where the frozen CTPR router corrects or harms predictions,
evaluates valid global-model reliability and selective prediction, and applies two
prespecified lexical-cue masks. It uses locked validation data only. It does not
train models, change `H`, `K`, or `tau`, benchmark efficiency, or open the test set.

## Runtime and scientific boundary

The notebook processes one canonical seed at a time on a Kaggle GPU. All results
are descriptive development-validation analyses. Operational labels are not clinical
diagnoses, and safety-relevant outputs do not support deployment claims.

In [1]:
import gc
import hashlib
import itertools
import json
import os
import random
import shutil
import zipfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / "phase11_mechanism_reliability_robustness_results"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir()
PREDICTION_DIR = OUTPUT_DIR / "predictions"
PREDICTION_DIR.mkdir()

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running Phase 11.")
DEVICE = torch.device("cuda")

LABELS = ("Suicidal", "Depressed", "Anxious", "Frustrated", "Others")
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
SEEDS = (13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407)
HARD_PAIRS = (
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others",
)
ALL_PAIRS = tuple("|".join(pair) for pair in itertools.combinations(LABELS, 2))
K_FIXED = 3
FROZEN_TAU = 0.10
ALWAYS_ROUTE_TAU = 1.00
TAU_GRID = (0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90, 1.00)
VALIDATION_SHA256 = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
PHASE10_ZIP_SHA256 = "7d9e0065632e024d4f9aece380a72cefec2b2bc3e03e18bf51e076229a421ff3"
HIGH_CONFIDENCE_THRESHOLD = 0.90
ECE_BINS = 15
NLL_EPSILON = 1e-12
MAX_LENGTH = 512
BATCH_SIZE = 16

DIRECT_WILDCARDS = ("suicid", "depress", "anxi", "frustrat")
OPERATIONAL_WILDCARDS = (
    "death", "sad", "anhedon", "letharg", "worthless",
    "hopeless", "helpless", "nervous", "panic", "fear", "dread",
    "annoy", "irritat", "dissatisf", "vex",
)
OPERATIONAL_EXACT_WORDS = (
    "dead", "dying", "empty", "emptiness", "miserable", "misery",
    "worry", "worries", "worried", "worrying", "afraid", "anger",
    "angry", "mad",
)
OPERATIONAL_PHRASES = (
    "kill myself", "end my life", "take my life", "want to die",
    "wish to die", "wish i were dead", "better off dead",
    "no reason to live", "low mood",
)
MASK_SPEC = {
    "version": "phase11-frozen-v1",
    "replacement_token": "<mask>",
    "case_sensitive": False,
    "direct_wildcards": DIRECT_WILDCARDS,
    "operational_wildcards": OPERATIONAL_WILDCARDS,
    "operational_exact_words": OPERATIONAL_EXACT_WORDS,
    "operational_phrases": OPERATIONAL_PHRASES,
    "self_harm_pattern": True,
    "operational_is_cumulative": True,
    "test_accessed": False,
}
(OUTPUT_DIR / "phase11_lexical_mask_spec.json").write_text(
    json.dumps(MASK_SPEC, indent=2, sort_keys=True) + "\n"
)
ACCESSED_INPUTS = []

print("CUDA device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Frozen H:", HARD_PAIRS)
print("Frozen K:", K_FIXED, "Frozen tau:", FROZEN_TAU)
print("PASS: lexical mask specification frozen before input discovery")
print("Test data accessed: False")

CUDA device: Tesla T4
PyTorch: 2.10.0+cu128
Frozen H: ('Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others')
Frozen K: 3 Frozen tau: 0.1
PASS: lexical mask specification frozen before input discovery
Test data accessed: False


## Required Kaggle inputs

Attach the locked validation dataset, Phase 8 specialist results, Phase 9 routing
results, the exact verified Phase 10 result ZIP, and the ten canonical global model
runs with their summary. Inputs are discovered recursively and verified before any
scientific output is accepted. Train, combined, and test datasets are forbidden.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError(f"Duplicate ZIP member: {archive_path}")
        for member_name in names:
            member_path = (destination / member_name).resolve()
            if not str(member_path).startswith(str(destination) + os.sep):
                raise ValueError(f"Unsafe ZIP member: {member_name}")
        archive.extractall(destination)


def write_deterministic_zip(source_dir, zip_path):
    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for source_path in sorted(
            path for path in source_dir.rglob("*") if path.is_file()
        ):
            archive_name = (
                source_dir.name
                + "/"
                + source_path.relative_to(source_dir).as_posix()
            )
            archive_info = zipfile.ZipInfo(
                archive_name,
                (2020, 1, 1, 0, 0, 0),
            )
            archive_info.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(archive_info, source_path.read_bytes())
    return sha256_file(zip_path)


def verify_manifest(root):
    manifest_path = root / "sha256_manifest.csv"
    frame = pd.read_csv(manifest_path)
    actual = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if set(frame["path"]) != actual or len(frame) != len(actual):
        raise ValueError(f"Manifest membership mismatch: {root}")
    for row in frame.itertuples():
        path = root / row.path
        if not path.is_file() or path.stat().st_size != int(row.bytes):
            raise ValueError(f"Manifest byte mismatch: {row.path}")
        if sha256_file(path) != row.sha256:
            raise ValueError(f"Manifest hash mismatch: {row.path}")
    return frame


def reject_forbidden_input(path):
    name = Path(path).name.lower()
    forbidden = ("_test", "combined", "train_locked")
    if any(marker in name for marker in forbidden):
        raise ValueError(f"Forbidden Phase 11 input selected: {path}")


def record_input(path, role, **extra):
    reject_forbidden_input(path)
    row = {
        "role": role,
        "path": str(path),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }
    row.update(extra)
    ACCESSED_INPUTS.append(row)


def find_input(filename, role, expected_sha256=None):
    matches = sorted(INPUT_ROOT.rglob(filename), key=str)
    if not matches:
        raise FileNotFoundError(f"Required Kaggle input not found: {filename}")
    path = matches[0]
    reject_forbidden_input(path)
    digest = sha256_file(path)
    if expected_sha256 is not None and digest != expected_sha256:
        raise ValueError(f"SHA-256 mismatch for {filename}: {digest}")
    record_input(path, role)
    return path


def resolve_result_dir(
    directory_name,
    archive_name,
    cache_name,
    expected_archive_sha256=None,
):
    directories = sorted(
        [path for path in INPUT_ROOT.rglob(directory_name) if path.is_dir()],
        key=str,
    )
    if directories:
        result_dir = directories[0]
    else:
        archives = sorted(INPUT_ROOT.rglob(archive_name), key=str)
        if not archives:
            raise FileNotFoundError(
                f"Attach {archive_name} or extracted {directory_name}"
            )
        archive_path = archives[0]
        reject_forbidden_input(archive_path)
        digest = sha256_file(archive_path)
        if (
            expected_archive_sha256 is not None
            and digest != expected_archive_sha256
        ):
            raise ValueError(
                f"Unexpected archive SHA-256 for {archive_name}: {digest}"
            )
        cache = WORKING_ROOT / cache_name
        if cache.exists():
            shutil.rmtree(cache)
        cache.mkdir(parents=True)
        safe_extract(archive_path, cache)
        result_dir = cache / directory_name
        record_input(archive_path, directory_name + "_archive")
    verify_manifest(result_dir)
    manifest_path = result_dir / "sha256_manifest.csv"
    record_input(manifest_path, directory_name + "_manifest")
    return result_dir


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")


def pair_slug(pair_name):
    first_label, second_label = pair_name.split("|")
    return first_label.lower() + "__" + second_label.lower()


print("PASS: path-safe input and integrity helpers initialized")

PASS: path-safe input and integrity helpers initialized


## Frozen lexical masks

The next cell embeds the locally tested Phase 11 scientific core. It implements the
fixed direct and cumulative operational masks, routing, correction/harm categories,
BCR, calibration, and empirical risk-coverage conventions.

In [3]:
import itertools
import re

import numpy as np


def _word_pattern(word):
    return rf"(?<![A-Za-z]){re.escape(word)}(?![A-Za-z])"


def _wildcard_pattern(stem):
    return rf"(?<![A-Za-z]){re.escape(stem)}[A-Za-z]*(?![A-Za-z])"


def _phrase_pattern(phrase):
    words = phrase.split()
    joined = r"\s+".join(re.escape(word) for word in words)
    return rf"(?<![A-Za-z]){joined}(?![A-Za-z])"


def compile_mask_patterns(mask_spec, mode):
    if mode not in {"direct", "operational"}:
        raise ValueError(f"Unknown lexical-mask mode: {mode}")
    entries = []
    if mode == "operational" and mask_spec.get("self_harm_pattern", False):
        entries.append(
            (
                "operational:self[- ]harm*",
                re.compile(
                    r"(?<![A-Za-z])self(?:-|\s+)harm[A-Za-z]*(?![A-Za-z])",
                    flags=re.IGNORECASE,
                ),
            )
        )
    if mode == "operational":
        phrases = sorted(
            mask_spec.get("operational_phrases", ()),
            key=lambda value: (-len(value), value),
        )
        for phrase in phrases:
            entries.append(
                (
                    f"operational:{phrase}",
                    re.compile(_phrase_pattern(phrase), flags=re.IGNORECASE),
                )
            )
    for stem in mask_spec.get("direct_wildcards", ()):
        entries.append(
            (
                f"direct:{stem}*",
                re.compile(_wildcard_pattern(stem), flags=re.IGNORECASE),
            )
        )
    if mode == "operational":
        for stem in mask_spec.get("operational_wildcards", ()):
            entries.append(
                (
                    f"operational:{stem}*",
                    re.compile(_wildcard_pattern(stem), flags=re.IGNORECASE),
                )
            )
        for word in mask_spec.get("operational_exact_words", ()):
            entries.append(
                (
                    f"operational:{word}",
                    re.compile(_word_pattern(word), flags=re.IGNORECASE),
                )
            )
    return tuple(entries)


def apply_lexical_mask(text, mask_spec, mode, mask_token="<mask>"):
    if not isinstance(text, str):
        raise TypeError("Lexical masking requires a string")
    candidates = []
    for priority, (cue_id, pattern) in enumerate(
        compile_mask_patterns(mask_spec, mode)
    ):
        for match in pattern.finditer(text):
            candidates.append(
                (
                    match.start(),
                    -(match.end() - match.start()),
                    priority,
                    match.end(),
                    cue_id,
                )
            )
    candidates.sort()
    selected = []
    current_end = -1
    for start, negative_length, priority, end, cue_id in candidates:
        del negative_length, priority
        if start < current_end:
            continue
        selected.append((start, end, cue_id))
        current_end = end
    if not selected:
        return {
            "text": text,
            "affected": False,
            "span_count": 0,
            "cue_ids": "",
            "original_characters": len(text),
            "masked_characters": len(text),
        }
    pieces = []
    cursor = 0
    cue_ids = []
    for start, end, cue_id in selected:
        pieces.append(text[cursor:start])
        pieces.append(f" {mask_token} ")
        cursor = end
        if cue_id not in cue_ids:
            cue_ids.append(cue_id)
    pieces.append(text[cursor:])
    masked = re.sub(r"\s+", " ", "".join(pieces)).strip()
    if not masked:
        masked = mask_token
    return {
        "text": masked,
        "affected": True,
        "span_count": len(selected),
        "cue_ids": ";".join(cue_ids),
        "original_characters": len(text),
        "masked_characters": len(masked),
    }


def route_hard_pairs(
    top2,
    margin,
    base,
    specialist_predictions,
    hard_pairs,
    tau,
    label_to_id,
):
    top2 = np.asarray(top2, dtype=int)
    margin = np.asarray(margin, dtype=float)
    final = np.asarray(base, dtype=int).copy()
    routed = np.zeros(len(final), dtype=bool)
    routed_pair = np.full(len(final), "", dtype=object)
    if float(tau) == 0.0:
        return final, routed, routed_pair
    id_to_label = {value: key for key, value in label_to_id.items()}
    label_order = dict(label_to_id)
    hard_set = set(hard_pairs)
    for row, (first, second) in enumerate(top2):
        pair_labels = sorted(
            (id_to_label[int(first)], id_to_label[int(second)]),
            key=label_order.__getitem__,
        )
        pair = "|".join(pair_labels)
        if pair in hard_set and float(margin[row]) <= float(tau):
            final[row] = int(specialist_predictions[pair][row])
            routed[row] = True
            routed_pair[row] = pair
    return final, routed, routed_pair


def confusion_matrix_fixed(y_true, y_pred, class_count):
    confusion = np.zeros((class_count, class_count), dtype=int)
    for truth, predicted in zip(
        np.asarray(y_true, dtype=int),
        np.asarray(y_pred, dtype=int),
    ):
        confusion[int(truth), int(predicted)] += 1
    return confusion


def classification_metrics(y_true, y_pred, class_count=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if len(y_true) == 0:
        raise ValueError("classification_metrics requires at least one row")
    confusion = confusion_matrix_fixed(y_true, y_pred, class_count)
    per_class = []
    f1_values = []
    supported_recalls = []
    for class_id in range(class_count):
        true_positive = int(confusion[class_id, class_id])
        predicted_count = int(confusion[:, class_id].sum())
        support = int(confusion[class_id, :].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (
            2.0 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        f1_values.append(f1)
        if support:
            supported_recalls.append(recall)
        per_class.append(
            {
                "class_id": class_id,
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "support": support,
            }
        )
    return {
        "accuracy": float(np.mean(y_true == y_pred)),
        "macro_f1": float(np.mean(f1_values)),
        "balanced_accuracy": float(np.mean(supported_recalls)),
        "represented_class_count": len(supported_recalls),
        "per_class": per_class,
        "confusion": confusion,
    }


def subset_metrics(y_true, y_pred, class_count=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if len(y_true) == 0:
        return {
            "defined": False,
            "missing_reason": "empty_subset",
            "row_count": 0,
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "balanced_accuracy": np.nan,
            "represented_class_count": 0,
        }
    metrics = classification_metrics(y_true, y_pred, class_count)
    return {
        "defined": True,
        "missing_reason": "",
        "row_count": len(y_true),
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "represented_class_count": metrics["represented_class_count"],
    }


def change_categories(y_true, global_pred, final_pred, active):
    y_true = np.asarray(y_true, dtype=int)
    global_pred = np.asarray(global_pred, dtype=int)
    final_pred = np.asarray(final_pred, dtype=int)
    active = np.asarray(active, dtype=bool)
    categories = np.full(len(y_true), "unchanged", dtype=object)
    changed = active & (global_pred != final_pred)
    categories[changed & (global_pred != y_true) & (final_pred == y_true)] = (
        "correction"
    )
    categories[changed & (global_pred == y_true) & (final_pred != y_true)] = "harm"
    categories[changed & (global_pred != y_true) & (final_pred != y_true)] = (
        "neutral_change"
    )
    return categories


def compute_bcr(confusion, labels):
    confusion = np.asarray(confusion, dtype=int)
    labels = tuple(labels)
    if confusion.shape != (len(labels), len(labels)):
        raise ValueError("Confusion matrix shape does not match labels")
    supports = confusion.sum(axis=1)
    rows = []
    for first, second in itertools.combinations(range(len(labels)), 2):
        first_to_second = int(confusion[first, second])
        second_to_first = int(confusion[second, first])
        first_rate = first_to_second / supports[first] if supports[first] else np.nan
        second_rate = second_to_first / supports[second] if supports[second] else np.nan
        bcr = 0.5 * (first_rate + second_rate)
        rows.append(
            {
                "pair": f"{labels[first]}|{labels[second]}",
                "first_support": int(supports[first]),
                "second_support": int(supports[second]),
                "first_to_second": first_to_second,
                "second_to_first": second_to_first,
                "pair_errors": first_to_second + second_to_first,
                "bcr": float(bcr),
            }
        )
    return rows


def ece_table(probabilities, y_true, n_bins=15):
    probabilities = np.asarray(probabilities, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    if probabilities.ndim != 2 or len(probabilities) != len(y_true):
        raise ValueError("Probability and label shapes do not align")
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = predicted == y_true
    assignments = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    rows = []
    ece = 0.0
    for bin_id in range(n_bins):
        selected = assignments == bin_id
        count = int(selected.sum())
        if count:
            mean_confidence = float(confidence[selected].mean())
            empirical_accuracy = float(correct[selected].mean())
            gap = abs(mean_confidence - empirical_accuracy)
            ece += (count / len(y_true)) * gap
        else:
            mean_confidence = np.nan
            empirical_accuracy = np.nan
            gap = np.nan
        rows.append(
            {
                "bin_id": bin_id,
                "lower": bin_id / n_bins,
                "upper": (bin_id + 1) / n_bins,
                "count": count,
                "mean_confidence": mean_confidence,
                "empirical_accuracy": empirical_accuracy,
                "absolute_gap": gap,
            }
        )
    return float(ece), rows


def multiclass_brier(probabilities, y_true, class_count=5):
    probabilities = np.asarray(probabilities, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    one_hot = np.eye(class_count, dtype=float)[y_true]
    if probabilities.shape != one_hot.shape:
        raise ValueError("Probability shape does not match class count")
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))


def multiclass_nll(probabilities, y_true, epsilon=1e-12):
    probabilities = np.asarray(probabilities, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    chosen = probabilities[np.arange(len(y_true)), y_true]
    return float(-np.mean(np.log(np.clip(chosen, epsilon, 1.0))))


def risk_coverage_curve(y_true, y_pred, score, original_row):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    score = np.asarray(score, dtype=float)
    original_row = np.asarray(original_row)
    if not (len(y_true) == len(y_pred) == len(score) == len(original_row)):
        raise ValueError("Risk-coverage arrays do not align")
    order = np.lexsort((original_row, -score))
    cumulative_errors = np.cumsum(y_true[order] != y_pred[order])
    total = len(order)
    rows = []
    for position, index in enumerate(order, start=1):
        rows.append(
            {
                "retained_n": position,
                "coverage": position / total,
                "risk": float(cumulative_errors[position - 1] / position),
                "score": float(score[index]),
                "original_row": original_row[index].item()
                if hasattr(original_row[index], "item")
                else original_row[index],
            }
        )
    return rows


def aurc(curve):
    if not curve:
        raise ValueError("AURC requires a nonempty curve")
    return float(np.mean([float(row["risk"]) for row in curve]))

## Lightweight protocol self-tests

These deterministic checks run before loading validation text or model outcomes.
They exercise path safety, lexical matching, routing, BCR, ECE, Brier, NLL, AURC,
and the zero-threshold bypass.

In [4]:
toy_spec = {
    "direct_wildcards": ("depress", "anxi"),
    "operational_wildcards": ("sad",),
    "operational_exact_words": ("dead",),
    "operational_phrases": ("kill myself",),
    "self_harm_pattern": True,
}
assert apply_lexical_mask("plain text", toy_spec, "direct")["text"] == "plain text"
toy_mask = apply_lexical_mask("DEPRESSED anxiety", toy_spec, "direct")
assert toy_mask["text"] == "<mask> <mask>" and toy_mask["span_count"] == 2
assert apply_lexical_mask(
    "I might kill myself", toy_spec, "operational"
)["text"] == "I might <mask>"
assert apply_lexical_mask(
    "self-harming", toy_spec, "operational"
)["text"] == "<mask>"

toy_labels = ("A", "B", "C")
toy_confusion = np.array([[8, 2, 0], [1, 9, 0], [0, 0, 10]])
toy_bcr = compute_bcr(toy_confusion, toy_labels)
assert np.isclose(toy_bcr[0]["bcr"], 0.15)

toy_probabilities = np.array([[0.8, 0.2], [0.4, 0.6], [0.9, 0.1]])
toy_true = np.array([0, 1, 1])
toy_ece, toy_bins = ece_table(toy_probabilities, toy_true, 2)
assert len(toy_bins) == 2 and np.isfinite(toy_ece)
assert np.isclose(
    multiclass_brier(toy_probabilities, toy_true, 2),
    np.mean([0.08, 0.32, 1.62]),
)
assert np.isclose(
    multiclass_nll(toy_probabilities, toy_true),
    -np.mean(np.log([0.8, 0.6, 0.1])),
)
toy_curve = risk_coverage_curve(
    np.array([0, 1, 1]),
    np.array([0, 0, 1]),
    np.array([0.8, 0.8, 0.2]),
    np.array([20, 10, 30]),
)
assert [row["original_row"] for row in toy_curve] == [10, 20, 30]
assert np.isclose(aurc(toy_curve), np.mean([1.0, 0.5, 1 / 3]))

toy_top2 = np.array([[0, 1], [1, 2]])
toy_base = toy_top2[:, 0]
toy_specialists = {"A|B": np.array([1, 0])}
toy_final, toy_routed, _ = route_hard_pairs(
    toy_top2,
    np.array([0.05, 0.01]),
    toy_base,
    toy_specialists,
    ("A|B",),
    0.10,
    {"A": 0, "B": 1, "C": 2},
)
assert toy_final.tolist() == [1, 1]
assert toy_routed.tolist() == [True, False]
zero_final, zero_routed, _ = route_hard_pairs(
    toy_top2,
    np.zeros(2),
    toy_base,
    toy_specialists,
    ("A|B",),
    0.00,
    {"A": 0, "B": 1, "C": 2},
)
assert np.array_equal(zero_final, toy_base) and not zero_routed.any()

helper_root = WORKING_ROOT / "phase11_helper_selftest"
if helper_root.exists():
    shutil.rmtree(helper_root)
helper_root.mkdir()
payload_path = helper_root / "payload.txt"
payload_path.write_text("phase11\n")
pd.DataFrame(
    [
        {
            "path": "payload.txt",
            "sha256": sha256_file(payload_path),
            "bytes": payload_path.stat().st_size,
        }
    ]
).to_csv(helper_root / "sha256_manifest.csv", index=False)
verify_manifest(helper_root)
unexpected_path = helper_root / "unexpected.txt"
unexpected_path.write_text("unexpected\n")
manifest_rejected = False
try:
    verify_manifest(helper_root)
except ValueError:
    manifest_rejected = True
assert manifest_rejected
unexpected_path.unlink()

helper_zip_one = WORKING_ROOT / "phase11_helper_one.zip"
helper_zip_two = WORKING_ROOT / "phase11_helper_two.zip"
digest_one = write_deterministic_zip(helper_root, helper_zip_one)
digest_two = write_deterministic_zip(helper_root, helper_zip_two)
assert digest_one == digest_two
with zipfile.ZipFile(helper_zip_one) as helper_archive:
    assert all(
        info.date_time == (2020, 1, 1, 0, 0, 0)
        for info in helper_archive.infolist()
    )

malicious_zip = WORKING_ROOT / "phase11_malicious.zip"
with zipfile.ZipFile(malicious_zip, "w") as malicious_archive:
    malicious_archive.writestr("../escape.txt", "blocked")
safe_rejected = False
try:
    safe_extract(malicious_zip, WORKING_ROOT / "phase11_bad_extract")
except ValueError:
    safe_rejected = True
assert safe_rejected
shutil.rmtree(helper_root)
helper_zip_one.unlink()
helper_zip_two.unlink()
malicious_zip.unlink()
print("PASS: archive and manifest helper self-tests")
print("PASS: lightweight Phase 11 helper tests")

PASS: archive and manifest helper self-tests
PASS: lightweight Phase 11 helper tests


## Verify locked validation and upstream phases

The cue specification already exists on disk. This section now loads the locked
validation split, verifies the complete Phase 8 specialist bank, freezes the Phase 9
router, and requires the audited Phase 10 execution record.

In [5]:
validation_path = find_input(
    "MentalDistress_validation_LOCKED.csv",
    "locked_validation",
    VALIDATION_SHA256,
)
validation = pd.read_csv(validation_path)
assert len(validation) == 1490
assert {"original_row", "text", "label"}.issubset(validation.columns)
assert validation["original_row"].is_unique
assert set(validation["label"]) == set(LABELS)
y_validation = validation["label"].map(LABEL_TO_ID).to_numpy(dtype=int)

phase8_dir = resolve_result_dir(
    "phase8_all_pair_specialists_results",
    "MentalDistress_Phase8_All_Pair_Specialists_Results.zip",
    "phase11_phase8_input",
)
phase8_protocol = json.loads((phase8_dir / "phase8_protocol.json").read_text())
assert phase8_protocol["phase"] == 8
assert phase8_protocol["head_count"] == 100
assert phase8_protocol["model_seeds"] == list(SEEDS)
assert phase8_protocol["pairs"] == list(ALL_PAIRS)
assert phase8_protocol["phase5_frozen_H"] == list(HARD_PAIRS)
assert phase8_protocol["phase5_frozen_K"] == K_FIXED
assert phase8_protocol["routing_performed"] is False
assert phase8_protocol["test_accessed"] is False
verified_head_count = 0
for model_seed in SEEDS:
    for pair_name in ALL_PAIRS:
        head_dir = (
            phase8_dir
            / "heads"
            / f"seed_{model_seed}"
            / pair_slug(pair_name)
        )
        assert (head_dir / "head.safetensors").is_file()
        assert (head_dir / "metadata.json").is_file()
        verified_head_count += 1
assert verified_head_count == 100

phase9_dir = resolve_result_dir(
    "phase9_routing_calibration_results",
    "MentalDistress_Phase9_Routing_Calibration_Results.zip",
    "phase11_phase9_input",
)
phase9_config = json.loads(
    (phase9_dir / "phase9_selected_config.json").read_text()
)
assert phase9_config["phase"] == 9
assert phase9_config["K_fixed"] == K_FIXED
assert phase9_config["H"] == list(HARD_PAIRS)
assert np.allclose(phase9_config["tau_grid"], TAU_GRID)
assert phase9_config["no_routing_baseline_included"] is True
assert np.isclose(phase9_config["selected_tau"], FROZEN_TAU)
assert phase9_config["selected_mode"] == "specialist_routing"
assert phase9_config["test_accessed"] is False

phase10_dir = resolve_result_dir(
    "phase10_controls_allocation_results",
    "MentalDistress_Phase10_Controls_and_Allocation_Results.zip",
    "phase11_phase10_input",
    PHASE10_ZIP_SHA256,
)
phase10_protocol = json.loads(
    (phase10_dir / "phase10_protocol.json").read_text()
)
phase10_execution = json.loads(
    (phase10_dir / "phase10_execution_summary.json").read_text()
)
assert phase10_protocol["phase"] == 10
assert phase10_protocol["labels"] == list(LABELS)
assert phase10_protocol["model_seeds"] == list(SEEDS)
assert phase10_protocol["H"] == list(HARD_PAIRS)
assert phase10_protocol["K"] == K_FIXED
assert np.isclose(phase10_protocol["frozen_tau"], FROZEN_TAU)
assert phase10_protocol["test_accessed"] is False
assert phase10_execution["status"] == "PASS"
assert phase10_execution["seed_count"] == 10
assert phase10_execution["phase9_reproduction_pass_count"] == 10
assert phase10_execution["control_seedwise_rows"] == 60
assert phase10_execution["allocation_seedwise_rows"] == 1200
assert phase10_execution["allocation_summary_rows"] == 120
assert phase10_execution["hard_allocation_rank"] == 3
assert np.isclose(phase10_execution["frozen_tau"], FROZEN_TAU)
assert phase10_execution["test_accessed"] is False

print("Validation rows:", len(validation))
print("Verified Phase 8 heads:", verified_head_count)
print("PASS: locked validation and Phase 8/9/10 artifacts verified")

Validation rows: 1490
Verified Phase 8 heads: 100
PASS: locked validation and Phase 8/9/10 artifacts verified


## Build frozen validation variants

The direct and cumulative operational masks are applied once. Unaffected rows remain
byte-identical to the original text. The audit table records cue identifiers and span
counts but never exports raw text.

In [6]:
validation_variants = {"original": validation.copy()}
mask_audit_rows = []
for variant, mode in (
    ("direct_mask", "direct"),
    ("operational_mask", "operational"),
):
    masked_text = []
    for row in validation.itertuples(index=False):
        result = apply_lexical_mask(
            str(row.text),
            MASK_SPEC,
            mode,
            MASK_SPEC["replacement_token"],
        )
        if not result["affected"]:
            assert result["text"] == str(row.text)
        masked_text.append(result["text"])
        mask_audit_rows.append(
            {
                "variant": variant,
                "original_row": row.original_row,
                "affected": result["affected"],
                "span_count": result["span_count"],
                "cue_ids": result["cue_ids"],
                "original_characters": result["original_characters"],
                "masked_characters": result["masked_characters"],
            }
        )
    variant_frame = validation.copy()
    variant_frame["text"] = masked_text
    validation_variants[variant] = variant_frame

mask_audit = pd.DataFrame(mask_audit_rows)
assert len(mask_audit) == 2 * len(validation)
assert set(validation_variants) == {
    "original", "direct_mask", "operational_mask"
}
coverage_rows = []
for variant, group in mask_audit.groupby("variant", sort=False):
    affected = int(group["affected"].sum())
    coverage_rows.append(
        {
            "variant": variant,
            "row_count": len(validation),
            "affected_rows": affected,
            "affected_percentage": 100.0 * affected / len(validation),
            "matched_spans": int(group["span_count"].sum()),
        }
    )
mask_coverage = pd.DataFrame(coverage_rows)
mask_coverage.to_csv(
    OUTPUT_DIR / "phase11_lexical_mask_coverage.csv",
    index=False,
)
mask_audit.to_csv(OUTPUT_DIR / "phase11_lexical_mask_audit.csv", index=False)
display(mask_coverage)
print("PASS: frozen lexical masks applied once across 1,490 rows")

,variant,row_count,affected_rows,affected_percentage,matched_spans
0,direct_mask,1490,432,28.993289,897
1,operational_mask,1490,771,51.744966,2040


PASS: frozen lexical masks applied once across 1,490 rows


## Verify canonical global checkpoints

Every model and validation-artifact manifest is verified. Exact seed matching is
mandatory because each Phase 8 head was trained on its corresponding global model's
representation space.

In [7]:
global_summary_path = find_input(
    "mentalroberta_full_10seed_summary.csv",
    "global_summary",
)
global_summary = pd.read_csv(global_summary_path)
assert sorted(global_summary["seed"].astype(int).tolist()) == list(SEEDS)

global_runs = {}
for summary_row in global_summary.itertuples():
    model_seed = int(summary_row.seed)
    candidates = [
        path
        for path in INPUT_ROOT.rglob(str(summary_row.run_id))
        if path.is_dir()
        and (path / "models/best_model/model.safetensors").is_file()
        and (path / "artifacts/validation_predictions.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"Complete global run missing: {summary_row.run_id}"
        )
    run_dir = sorted(candidates, key=str)[0]
    verify_manifest(run_dir / "models")
    verify_manifest(run_dir / "artifacts")
    global_runs[model_seed] = run_dir
    for role, manifest_path in (
        ("global_model_manifest", run_dir / "models/sha256_manifest.csv"),
        ("global_artifact_manifest", run_dir / "artifacts/sha256_manifest.csv"),
    ):
        record_input(manifest_path, role, model_seed=model_seed)
    print(f"  seed {model_seed}: {run_dir.name}")
assert sorted(global_runs) == list(SEEDS)
print("PASS: ten canonical global checkpoints verified")

  seed 13: mentalroberta_full_seed13_len512_fea2991778f3
  seed 42: mentalroberta_full_seed42_len512_be72f916ca15
  seed 77: mentalroberta_full_seed77_len512_d14375990239
  seed 123: mentalroberta_full_seed123_len512_d1ad313ee75d
  seed 211: mentalroberta_full_seed211_len512_149308113b40
  seed 340: mentalroberta_full_seed340_len512_6353292c8230
  seed 512: mentalroberta_full_seed512_len512_412e1caa7392
  seed 1024: mentalroberta_full_seed1024_len512_e91f034d8a71
  seed 2026: mentalroberta_full_seed2026_len512_f6630d942394
  seed 3407: mentalroberta_full_seed3407_len512_d8596ea710c3
PASS: ten canonical global checkpoints verified


## Frozen model, specialist, and analysis helpers

Global checkpoints run in float32 with dynamic padding and maximum length 512.
Only frozen seed-matched hard-pair heads are applied. Helper functions create
complete overall, per-class, subset, and row-level records without raw text.

In [8]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def encode_frame(tokenizer, model, frame, batch_size=BATCH_SIZE):
    feature_batches = []
    logit_batches = []
    texts = frame["text"].astype(str).tolist()
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            encoded = tokenizer(
                texts[start : start + batch_size],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            encoded = {
                name: tensor.to(DEVICE) for name, tensor in encoded.items()
            }
            output = model(
                **encoded,
                output_hidden_states=True,
                return_dict=True,
            )
            feature_batches.append(
                output.hidden_states[-1][:, 0, :].float().cpu()
            )
            logit_batches.append(output.logits.float().cpu())
    return torch.cat(feature_batches), torch.cat(logit_batches)


def load_hard_specialist_predictions(model_seed, features):
    predictions = {}
    feature_array = features.numpy()
    for pair_name in HARD_PAIRS:
        head_dir = (
            phase8_dir
            / "heads"
            / f"seed_{model_seed}"
            / pair_slug(pair_name)
        )
        metadata = json.loads((head_dir / "metadata.json").read_text())
        assert metadata["phase"] == 8
        assert metadata["model_seed"] == model_seed
        assert metadata["pair"] == pair_name
        assert metadata["status"] == "PASS"
        assert metadata["test_accessed"] is False
        tensors = load_file(str(head_dir / "head.safetensors"), device="cpu")
        weight = tensors["classifier.weight"].numpy()
        bias = tensors["classifier.bias"].numpy()
        assert weight.shape == (2, features.shape[1])
        head_logits = feature_array @ weight.T + bias
        first_label, second_label = pair_name.split("|")
        predictions[pair_name] = np.where(
            head_logits.argmax(axis=1) == 0,
            LABEL_TO_ID[first_label],
            LABEL_TO_ID[second_label],
        )
    return predictions


def overall_metric_record(model_seed, system_id, variant, y_true, prediction):
    result = classification_metrics(y_true, prediction, len(LABELS))
    return {
        "model_seed": model_seed,
        "system_id": system_id,
        "variant": variant,
        "row_count": len(y_true),
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "balanced_accuracy": result["balanced_accuracy"],
    }


def per_class_records(model_seed, system_id, variant, y_true, prediction):
    result = classification_metrics(y_true, prediction, len(LABELS))
    rows = []
    for record in result["per_class"]:
        rows.append(
            {
                "model_seed": model_seed,
                "system_id": system_id,
                "variant": variant,
                "class_id": record["class_id"],
                "class_label": LABELS[record["class_id"]],
                "precision": record["precision"],
                "recall": record["recall"],
                "f1": record["f1"],
                "support": record["support"],
            }
        )
    return rows


def distribution_record(values, prefix):
    values = np.asarray(values, dtype=float)
    record = {prefix + "_count": len(values)}
    if len(values) == 0:
        for name in ("mean", "sd", "min", "q1", "median", "q3", "max"):
            record[prefix + "_" + name] = np.nan
        return record
    record[prefix + "_mean"] = float(values.mean())
    record[prefix + "_sd"] = (
        float(values.std(ddof=1)) if len(values) > 1 else 0.0
    )
    record[prefix + "_min"] = float(values.min())
    record[prefix + "_q1"] = float(np.quantile(values, 0.25))
    record[prefix + "_median"] = float(np.quantile(values, 0.50))
    record[prefix + "_q3"] = float(np.quantile(values, 0.75))
    record[prefix + "_max"] = float(values.max())
    return record


def read_bool(series):
    if series.dtype == bool:
        return series.to_numpy(dtype=bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    return series.astype(str).str.lower().map(mapping).to_numpy(dtype=bool)


def save_seed_predictions(
    model_seed,
    variant,
    probabilities,
    top2,
    margin,
    flat,
    conditional,
    routed,
    routed_pair,
    always=None,
    always_routed=None,
    always_pair=None,
):
    payload = {
        "original_row": validation["original_row"],
        "true_label": validation["label"],
        "global_id": flat,
        "conditional_id": conditional,
        "routed": routed,
        "routed_pair": routed_pair,
        "top1": top2[:, 0],
        "top2": top2[:, 1],
        "margin": margin,
        "global_confidence": probabilities.max(axis=1),
    }
    for class_id, label in enumerate(LABELS):
        payload[f"probability_{label.lower()}"] = probabilities[:, class_id]
    if always is not None:
        payload["always_route_id"] = always
        payload["always_routed"] = always_routed
        payload["always_routed_pair"] = always_pair
    path = PREDICTION_DIR / f"{variant}_seed{model_seed}.csv"
    pd.DataFrame(payload).to_csv(path, index=False)


def summarize_frame(frame, group_columns, value_columns):
    aggregations = {}
    for column in value_columns:
        aggregations["mean_" + column] = (column, "mean")
        aggregations["sd_" + column] = (column, "std")
    return frame.groupby(group_columns, as_index=False).agg(**aggregations)


print("PASS: frozen inference and analysis helpers initialized")

PASS: frozen inference and analysis helpers initialized


## Phase 10 reproduction gate

Original-text inference must reproduce all verified Phase 10 A0, A1, and A2 outputs
before any Phase 11 analysis is accepted. The same loop then evaluates both frozen
lexical challenges without retraining any component.

In [9]:
reproduction_rows = []
original_metric_rows = []
original_per_class_rows = []
correction_seedwise_rows = []
correction_pair_rows = []
correction_class_rows = []
mechanism_case_rows = []
confidence_margin_rows = []
high_confidence_rows = []
routed_subset_rows = []
bcr_rows = []
suicidal_rows = []
reliability_rows = []
reliability_bin_rows = []
risk_coverage_rows = []
aurc_rows = []
lexical_metric_rows = []
lexical_per_class_rows = []
routing_change_rows = []

variant_affected = {}
for variant in ("direct_mask", "operational_mask"):
    audit = mask_audit[mask_audit["variant"] == variant]
    audit = audit.set_index("original_row").loc[
        validation["original_row"]
    ]
    variant_affected[variant] = audit["affected"].to_numpy(dtype=bool)

for seed_index, model_seed in enumerate(SEEDS, start=1):
    print(f"[{seed_index}/{len(SEEDS)}] seed {model_seed}")
    seed_everything(model_seed)
    model_dir = global_runs[model_seed] / "models/best_model"
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        output_hidden_states=True,
    )
    assert model.config.num_labels == len(LABELS)
    model = model.eval().to(DEVICE)
    seed_outputs = {}

    for variant in ("original", "direct_mask", "operational_mask"):
        print("  inference:", variant)
        features, logits = encode_frame(
            tokenizer,
            model,
            validation_variants[variant],
        )
        probabilities = torch.softmax(logits, dim=1).numpy()
        assert probabilities.shape == (len(validation), len(LABELS))
        assert np.isfinite(probabilities).all()
        assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
        top2 = np.argsort(-probabilities, axis=1)[:, :2]
        row_index = np.arange(len(validation))
        margin = (
            probabilities[row_index, top2[:, 0]]
            - probabilities[row_index, top2[:, 1]]
        )
        flat = top2[:, 0]
        specialist_predictions = load_hard_specialist_predictions(
            model_seed,
            features,
        )
        conditional, routed, routed_pair = route_hard_pairs(
            top2,
            margin,
            flat,
            specialist_predictions,
            HARD_PAIRS,
            FROZEN_TAU,
            LABEL_TO_ID,
        )
        if variant == "original":
            always, always_routed, always_pair = route_hard_pairs(
                top2,
                margin,
                flat,
                specialist_predictions,
                HARD_PAIRS,
                ALWAYS_ROUTE_TAU,
                LABEL_TO_ID,
            )
        else:
            always = None
            always_routed = None
            always_pair = None
        seed_outputs[variant] = {
            "probabilities": probabilities,
            "top2": top2,
            "margin": margin,
            "confidence": probabilities.max(axis=1),
            "flat": flat,
            "conditional": conditional,
            "routed": routed,
            "routed_pair": routed_pair,
            "always": always,
            "always_routed": always_routed,
            "always_pair": always_pair,
        }
        del features, logits, specialist_predictions
        gc.collect()

    original = seed_outputs["original"]
    reproduction_record = {
        "model_seed": model_seed,
        "A0_exact": True,
        "A1_exact": True,
        "A2_exact": True,
        "routing_exact": True,
        "pair_exact": True,
        "top2_exact": True,
        "margin_max_abs_diff": 0.0,
        "status": "PASS",
    }
    margin_differences = []
    for control_id, final, active, pair in (
        (
            "A0_flat",
            original["flat"],
            np.zeros(len(validation), dtype=bool),
            np.full(len(validation), "", dtype=object),
        ),
        (
            "A1_conditional",
            original["conditional"],
            original["routed"],
            original["routed_pair"],
        ),
        (
            "A2_always_route",
            original["always"],
            original["always_routed"],
            original["always_pair"],
        ),
    ):
        saved_path = (
            phase10_dir
            / "controls"
            / control_id
            / f"seed_{model_seed}"
            / "validation_predictions.csv"
        )
        saved = pd.read_csv(saved_path)
        assert len(saved) == len(validation)
        assert np.array_equal(
            saved["original_row"].to_numpy(),
            validation["original_row"].to_numpy(),
        )
        assert np.array_equal(
            saved["true_label"].to_numpy(),
            validation["label"].to_numpy(),
        )
        assert np.array_equal(saved["global_id"].to_numpy(), original["flat"])
        assert np.array_equal(saved["final_id"].to_numpy(), final)
        assert np.array_equal(read_bool(saved["active"]), active)
        assert np.array_equal(
            saved["routed_pair"].fillna("").to_numpy(),
            pair,
        )
        assert np.array_equal(saved["top1"].to_numpy(), original["top2"][:, 0])
        assert np.array_equal(saved["top2"].to_numpy(), original["top2"][:, 1])
        difference = float(
            np.max(np.abs(saved["margin"].to_numpy() - original["margin"]))
        )
        assert difference <= 1e-6
        margin_differences.append(difference)
    reproduction_record["margin_max_abs_diff"] = max(margin_differences)
    reproduction_rows.append(reproduction_record)
    save_seed_predictions(
        model_seed,
        "original",
        original["probabilities"],
        original["top2"],
        original["margin"],
        original["flat"],
        original["conditional"],
        original["routed"],
        original["routed_pair"],
        original["always"],
        original["always_routed"],
        original["always_pair"],
    )
    for variant in ("direct_mask", "operational_mask"):
        masked = seed_outputs[variant]
        save_seed_predictions(
            model_seed,
            variant,
            masked["probabilities"],
            masked["top2"],
            masked["margin"],
            masked["flat"],
            masked["conditional"],
            masked["routed"],
            masked["routed_pair"],
        )

    original_systems = {
        "A0_flat": original["flat"],
        "A1_conditional": original["conditional"],
        "A2_always_route": original["always"],
    }
    for system_id, prediction in original_systems.items():
        original_metric_rows.append(
            overall_metric_record(
                model_seed,
                system_id,
                "original",
                y_validation,
                prediction,
            )
        )
        original_per_class_rows.extend(
            per_class_records(
                model_seed,
                system_id,
                "original",
                y_validation,
                prediction,
            )
        )
        confusion = classification_metrics(
            y_validation,
            prediction,
            len(LABELS),
        )["confusion"]
        for row in compute_bcr(confusion, LABELS):
            bcr_rows.append(
                {
                    "model_seed": model_seed,
                    "system_id": system_id,
                    **row,
                }
            )

        suicidal_class = classification_metrics(
            y_validation,
            prediction,
            len(LABELS),
        )["per_class"][LABEL_TO_ID["Suicidal"]]
        suicidal_true = y_validation == LABEL_TO_ID["Suicidal"]
        suicidal_fn = suicidal_true & (
            prediction != LABEL_TO_ID["Suicidal"]
        )
        suicidal_record = {
            "model_seed": model_seed,
            "system_id": system_id,
            "precision": suicidal_class["precision"],
            "recall": suicidal_class["recall"],
            "f1": suicidal_class["f1"],
            "support": suicidal_class["support"],
            "false_negatives": int(suicidal_fn.sum()),
        }
        for destination_id, destination in enumerate(LABELS):
            if destination == "Suicidal":
                continue
            key = "suicidal_to_" + destination.lower()
            suicidal_record[key] = int(
                np.sum(suicidal_true & (prediction == destination_id))
            )
        if system_id == "A0_flat":
            suicidal_record.update(
                corrected_false_negatives=0,
                introduced_false_negatives=0,
                unchanged_false_negatives=int(suicidal_fn.sum()),
            )
        else:
            suicidal_record.update(
                corrected_false_negatives=int(
                    np.sum(
                        suicidal_true
                        & (original["flat"] != LABEL_TO_ID["Suicidal"])
                        & (prediction == LABEL_TO_ID["Suicidal"])
                    )
                ),
                introduced_false_negatives=int(
                    np.sum(
                        suicidal_true
                        & (original["flat"] == LABEL_TO_ID["Suicidal"])
                        & (prediction != LABEL_TO_ID["Suicidal"])
                    )
                ),
                unchanged_false_negatives=int(
                    np.sum(
                        suicidal_true
                        & (original["flat"] != LABEL_TO_ID["Suicidal"])
                        & (prediction != LABEL_TO_ID["Suicidal"])
                    )
                ),
            )
        suicidal_rows.append(suicidal_record)

    for system_id, final, active, routed_pair in (
        (
            "A1_conditional",
            original["conditional"],
            original["routed"],
            original["routed_pair"],
        ),
        (
            "A2_always_route",
            original["always"],
            original["always_routed"],
            original["always_pair"],
        ),
    ):
        categories = change_categories(
            y_validation,
            original["flat"],
            final,
            active,
        )
        counts = {
            category: int(np.sum(categories == category))
            for category in (
                "correction", "harm", "neutral_change", "unchanged"
            )
        }
        correction_seedwise_rows.append(
            {
                "model_seed": model_seed,
                "system_id": system_id,
                **counts,
                "unchanged_routed": int(
                    np.sum(active & (categories == "unchanged"))
                ),
                "routed_n": int(active.sum()),
                "net_corrections": counts["correction"] - counts["harm"],
            }
        )
        for pair_name in HARD_PAIRS:
            selected = active & (routed_pair == pair_name)
            pair_counts = {
                category: int(np.sum(selected & (categories == category)))
                for category in (
                    "correction", "harm", "neutral_change", "unchanged"
                )
            }
            correction_pair_rows.append(
                {
                    "model_seed": model_seed,
                    "system_id": system_id,
                    "pair": pair_name,
                    **pair_counts,
                    "routed_n": int(selected.sum()),
                    "net_corrections": (
                        pair_counts["correction"] - pair_counts["harm"]
                    ),
                }
            )
        for class_id, class_label in enumerate(LABELS):
            selected = active & (y_validation == class_id)
            class_counts = {
                category: int(np.sum(selected & (categories == category)))
                for category in (
                    "correction", "harm", "neutral_change", "unchanged"
                )
            }
            correction_class_rows.append(
                {
                    "model_seed": model_seed,
                    "system_id": system_id,
                    "true_class": class_label,
                    **class_counts,
                    "routed_n": int(selected.sum()),
                    "net_corrections": (
                        class_counts["correction"] - class_counts["harm"]
                    ),
                }
            )

        for row_index, original_row in enumerate(
            validation["original_row"].to_numpy()
        ):
            mechanism_case_rows.append(
                {
                    "model_seed": model_seed,
                    "system_id": system_id,
                    "original_row": original_row,
                    "true_id": int(y_validation[row_index]),
                    "global_id": int(original["flat"][row_index]),
                    "final_id": int(final[row_index]),
                    "active": bool(active[row_index]),
                    "routed_pair": routed_pair[row_index],
                    "category": categories[row_index],
                    "global_confidence": original["confidence"][row_index],
                    "margin": original["margin"][row_index],
                }
            )
        for category in (
            "correction", "harm", "neutral_change", "unchanged"
        ):
            selected = categories == category
            if category == "unchanged":
                selected &= active
            record = {
                "model_seed": model_seed,
                "system_id": system_id,
                "category": category,
            }
            record.update(
                distribution_record(
                    original["confidence"][selected],
                    "confidence",
                )
            )
            record.update(
                distribution_record(original["margin"][selected], "margin")
            )
            confidence_margin_rows.append(record)

        for subset_name, selected in (
            ("routed", active),
            ("non_routed", ~active),
        ):
            for evaluated_system, prediction in (
                ("A0_flat_reference", original["flat"]),
                (system_id, final),
            ):
                routed_subset_rows.append(
                    {
                        "model_seed": model_seed,
                        "partition_system": system_id,
                        "evaluated_system": evaluated_system,
                        "subset": subset_name,
                        **subset_metrics(
                            y_validation[selected],
                            prediction[selected],
                            len(LABELS),
                        ),
                    }
                )

    high_error = (
        (original["flat"] != y_validation)
        & (
            original["confidence"]
            >= HIGH_CONFIDENCE_THRESHOLD
        )
    )
    for row_index in np.flatnonzero(high_error):
        high_confidence_rows.append(
            {
                "model_seed": model_seed,
                "original_row": validation.iloc[row_index]["original_row"],
                "true_id": int(y_validation[row_index]),
                "true_label": LABELS[y_validation[row_index]],
                "global_id": int(original["flat"][row_index]),
                "global_label": LABELS[original["flat"][row_index]],
                "global_confidence": original["confidence"][row_index],
                "margin": original["margin"][row_index],
                "A1_routed": bool(original["routed"][row_index]),
                "A1_final_id": int(original["conditional"][row_index]),
                "A1_corrected": bool(
                    original["conditional"][row_index]
                    == y_validation[row_index]
                ),
                "A2_routed": bool(original["always_routed"][row_index]),
                "A2_final_id": int(original["always"][row_index]),
                "A2_corrected": bool(
                    original["always"][row_index] == y_validation[row_index]
                ),
            }
        )

    ece, bin_rows = ece_table(
        original["probabilities"],
        y_validation,
        ECE_BINS,
    )
    one_hot = np.eye(len(LABELS))[y_validation]
    reliability_rows.append(
        {
            "model_seed": model_seed,
            "ece": ece,
            "brier": multiclass_brier(
                original["probabilities"],
                y_validation,
                len(LABELS),
            ),
            "nll": multiclass_nll(
                original["probabilities"],
                y_validation,
                NLL_EPSILON,
            ),
            "macro_roc_auc": roc_auc_score(
                one_hot,
                original["probabilities"],
                average="macro",
                multi_class="ovr",
            ),
            "macro_pr_auc": average_precision_score(
                one_hot,
                original["probabilities"],
                average="macro",
            ),
        }
    )
    for row in bin_rows:
        reliability_bin_rows.append({"model_seed": model_seed, **row})

    for system_id, prediction in original_systems.items():
        curve = risk_coverage_curve(
            y_validation,
            prediction,
            original["confidence"],
            validation["original_row"].to_numpy(),
        )
        for row in curve:
            risk_coverage_rows.append(
                {
                    "model_seed": model_seed,
                    "system_id": system_id,
                    **row,
                }
            )
        aurc_rows.append(
            {
                "model_seed": model_seed,
                "system_id": system_id,
                "aurc": aurc(curve),
                "ranking_score": "global_max_softmax",
            }
        )

    for variant, output in seed_outputs.items():
        affected = (
            np.zeros(len(validation), dtype=bool)
            if variant == "original"
            else variant_affected[variant]
        )
        for system_id, prediction in (
            ("A0_flat", output["flat"]),
            ("A1_conditional", output["conditional"]),
        ):
            record = overall_metric_record(
                model_seed,
                system_id,
                variant,
                y_validation,
                prediction,
            )
            record.update(
                routing_coverage=(
                    0.0
                    if system_id == "A0_flat"
                    else float(output["routed"].mean())
                ),
                routed_n=(
                    0
                    if system_id == "A0_flat"
                    else int(output["routed"].sum())
                ),
                affected_rows=int(affected.sum()),
            )
            for subset_name, selected in (
                ("affected", affected),
                ("unaffected", ~affected),
            ):
                subset = subset_metrics(
                    y_validation[selected],
                    prediction[selected],
                    len(LABELS),
                )
                for metric in (
                    "row_count", "accuracy", "macro_f1",
                    "balanced_accuracy", "represented_class_count",
                ):
                    record[f"{subset_name}_{metric}"] = subset[metric]
                record[f"{subset_name}_defined"] = subset["defined"]
                record[f"{subset_name}_missing_reason"] = subset[
                    "missing_reason"
                ]
            lexical_metric_rows.append(record)
            lexical_per_class_rows.extend(
                per_class_records(
                    model_seed,
                    system_id,
                    variant,
                    y_validation,
                    prediction,
                )
            )

    for variant in ("direct_mask", "operational_mask"):
        masked = seed_outputs[variant]
        original_routed = original["routed"]
        masked_routed = masked["routed"]
        both = original_routed & masked_routed
        routing_change_rows.append(
            {
                "model_seed": model_seed,
                "variant": variant,
                "original_routed_n": int(original_routed.sum()),
                "masked_routed_n": int(masked_routed.sum()),
                "route_entries": int((~original_routed & masked_routed).sum()),
                "route_exits": int((original_routed & ~masked_routed).sum()),
                "routed_pair_changes": int(
                    np.sum(
                        both
                        & (
                            original["routed_pair"]
                            != masked["routed_pair"]
                        )
                    )
                ),
                "routing_indicator_changes": int(
                    np.sum(original_routed != masked_routed)
                ),
            }
        )

    del model, tokenizer, seed_outputs
    gc.collect()
    torch.cuda.empty_cache()
    print("  PASS: Phase 10 reproduced and three variants analyzed")

phase10_reproduction = pd.DataFrame(reproduction_rows)
assert len(phase10_reproduction) == len(SEEDS)
assert phase10_reproduction["status"].eq("PASS").all()
print("PASS: all ten Phase 10 A0/A1/A2 outputs reproduced")

[1/10] seed 13


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  inference: original
  inference: direct_mask
  inference: operational_mask
  PASS: Phase 10 reproduced and three variants analyzed
[2/10] seed 42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  inference: original
  inference: direct_mask
  inference: operational_mask
  PASS: Phase 10 reproduced and three variants analyzed
[3/10] seed 77


...

  inference: original
  inference: direct_mask
  inference: operational_mask
  PASS: Phase 10 reproduced and three variants analyzed
[9/10] seed 2026


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  inference: original
  inference: direct_mask
  inference: operational_mask
  PASS: Phase 10 reproduced and three variants analyzed
[10/10] seed 3407


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  inference: original
  inference: direct_mask
  inference: operational_mask
  PASS: Phase 10 reproduced and three variants analyzed
PASS: all ten Phase 10 A0/A1/A2 outputs reproduced


## Mechanism and Suicidal error analysis

This section aggregates corrections, harms, neutral changes, routed subsets, BCR,
high-confidence global errors, and Suicidal false-negative destinations. Equal seed
weighting is maintained throughout.

In [10]:
original_seedwise = pd.DataFrame(original_metric_rows)
original_per_class = pd.DataFrame(original_per_class_rows)
correction_seedwise = pd.DataFrame(correction_seedwise_rows)
correction_by_pair = pd.DataFrame(correction_pair_rows)
correction_by_class = pd.DataFrame(correction_class_rows)
mechanism_cases = pd.DataFrame(mechanism_case_rows)
confidence_margin_summary = pd.DataFrame(confidence_margin_rows)
high_confidence_errors = pd.DataFrame(
    high_confidence_rows,
    columns=[
        "model_seed", "original_row", "true_id", "true_label",
        "global_id", "global_label", "global_confidence", "margin",
        "A1_routed", "A1_final_id", "A1_corrected",
        "A2_routed", "A2_final_id", "A2_corrected",
    ],
)
routed_subset_metrics = pd.DataFrame(routed_subset_rows)
bcr_seedwise = pd.DataFrame(bcr_rows)
suicidal_seedwise = pd.DataFrame(suicidal_rows)

assert len(original_seedwise) == len(SEEDS) * 3
assert len(correction_seedwise) == len(SEEDS) * 2
assert len(correction_by_pair) == len(SEEDS) * 2 * len(HARD_PAIRS)
assert len(correction_by_class) == len(SEEDS) * 2 * len(LABELS)
assert len(bcr_seedwise) == len(SEEDS) * 3 * len(ALL_PAIRS)
assert len(suicidal_seedwise) == len(SEEDS) * 3

original_summary = summarize_frame(
    original_seedwise,
    ["system_id", "variant"],
    ["accuracy", "macro_f1", "balanced_accuracy"],
)
correction_summary = summarize_frame(
    correction_seedwise,
    ["system_id"],
    [
        "correction", "harm", "neutral_change", "unchanged_routed",
        "routed_n", "net_corrections",
    ],
)
pair_summary = summarize_frame(
    correction_by_pair,
    ["system_id", "pair"],
    ["correction", "harm", "neutral_change", "routed_n", "net_corrections"],
)
class_summary = summarize_frame(
    correction_by_class,
    ["system_id", "true_class"],
    ["correction", "harm", "neutral_change", "routed_n", "net_corrections"],
)

bcr_baseline = bcr_seedwise[
    bcr_seedwise["system_id"] == "A0_flat"
][["model_seed", "pair", "bcr"]].rename(columns={"bcr": "baseline_bcr"})
bcr_seedwise = bcr_seedwise.merge(
    bcr_baseline,
    on=["model_seed", "pair"],
    how="left",
)
bcr_seedwise["absolute_change_vs_A0"] = (
    bcr_seedwise["bcr"] - bcr_seedwise["baseline_bcr"]
)
bcr_seedwise["relative_change_vs_A0"] = np.where(
    bcr_seedwise["baseline_bcr"] == 0,
    np.nan,
    bcr_seedwise["absolute_change_vs_A0"]
    / bcr_seedwise["baseline_bcr"],
)
bcr_summary = summarize_frame(
    bcr_seedwise,
    ["system_id", "pair"],
    ["bcr", "absolute_change_vs_A0", "relative_change_vs_A0"],
)

suicidal_value_columns = [
    "precision", "recall", "f1", "false_negatives",
    "suicidal_to_depressed", "suicidal_to_anxious",
    "suicidal_to_frustrated", "suicidal_to_others",
    "corrected_false_negatives", "introduced_false_negatives",
    "unchanged_false_negatives",
]
suicidal_summary = summarize_frame(
    suicidal_seedwise,
    ["system_id"],
    suicidal_value_columns,
)

original_seedwise.to_csv(
    OUTPUT_DIR / "phase11_original_seedwise_metrics.csv",
    index=False,
)
original_summary.to_csv(
    OUTPUT_DIR / "phase11_original_metric_summary.csv",
    index=False,
)
correction_seedwise.to_csv(
    OUTPUT_DIR / "phase11_correction_harm_seedwise.csv",
    index=False,
)
correction_by_pair.to_csv(
    OUTPUT_DIR / "phase11_correction_harm_by_pair.csv",
    index=False,
)
correction_by_class.to_csv(
    OUTPUT_DIR / "phase11_correction_harm_by_class.csv",
    index=False,
)
mechanism_cases.to_csv(
    OUTPUT_DIR / "phase11_mechanism_cases.csv",
    index=False,
)
confidence_margin_summary.to_csv(
    OUTPUT_DIR / "phase11_confidence_margin_summary.csv",
    index=False,
)
high_confidence_errors.to_csv(
    OUTPUT_DIR / "phase11_high_confidence_errors.csv",
    index=False,
)
routed_subset_metrics.to_csv(
    OUTPUT_DIR / "phase11_routed_subset_metrics.csv",
    index=False,
)
bcr_seedwise.to_csv(OUTPUT_DIR / "phase11_bcr_seedwise.csv", index=False)
bcr_summary.to_csv(OUTPUT_DIR / "phase11_bcr_summary.csv", index=False)
suicidal_seedwise.to_csv(
    OUTPUT_DIR / "phase11_suicidal_seedwise.csv",
    index=False,
)
suicidal_summary.to_csv(
    OUTPUT_DIR / "phase11_suicidal_summary.csv",
    index=False,
)
display(original_summary.round(6))
display(correction_summary.round(3))
display(bcr_summary[bcr_summary["pair"].isin(HARD_PAIRS)].round(6))
display(suicidal_summary.round(4))
print("PASS: mechanism, BCR, and Suicidal analyses aggregated")

,system_id,variant,mean_accuracy,sd_accuracy,mean_macro_f1,sd_macro_f1,mean_balanced_accuracy,sd_balanced_accuracy
0,A0_flat,original,0.866577,0.003315,0.863198,0.003567,0.862223,0.00377
1,A1_conditional,original,0.867517,0.004066,0.864138,0.004309,0.863153,0.00447
2,A2_always_route,original,0.868255,0.003580,0.864921,0.004024,0.864052,0.00455


,system_id,mean_correction,sd_correction,mean_harm,sd_harm,mean_neutral_change,sd_neutral_change,mean_unchanged_routed,sd_unchanged_routed,mean_routed_n,sd_routed_n,mean_net_corrections,sd_net_corrections
0,A1_conditional,2.6,2.119,1.2,1.687,0.3,0.949,5.7,3.860,9.8,6.477,1.4,1.776
1,A2_always_route,8.3,4.498,5.8,4.185,2.1,1.792,813.1,237.172,829.3,242.560,2.5,2.759


,system_id,pair,mean_bcr,sd_bcr,mean_absolute_change_vs_A0,sd_absolute_change_vs_A0,mean_relative_change_vs_A0,sd_relative_change_vs_A0
4,A0_flat,Depressed|Others,0.046388,0.006627,0.000000,0.000000,0.000000,0.000000
5,A0_flat,Frustrated|Others,0.093227,0.004661,0.000000,0.000000,0.000000,0.000000
7,A0_flat,Suicidal|Depressed,0.078162,0.004147,0.000000,0.000000,0.000000,0.000000
14,A1_conditional,Depressed|Others,0.045932,0.006615,-0.000457,0.001379,-0.009526,0.030684
15,A1_conditional,Frustrated|Others,0.092029,0.005881,-0.001198,0.001574,-0.013383,0.016877
17,A1_conditional,Suicidal|Depressed,0.077849,0.005042,-0.000313,0.001791,-0.004349,0.022203
24,A2_always_route,Depressed|Others,0.047107,0.006267,0.000719,0.002901,0.019258,0.067666
25,A2_always_route,Frustrated|Others,0.091356,0.004669,-0.001870,0.002818,-0.019753,0.029593
27,A2_always_route,Suicidal|Depressed,0.075651,0.002993,-0.002511,0.003280,-0.030866,0.039237


,system_id,mean_precision,sd_precision,mean_recall,sd_recall,mean_f1,sd_f1,mean_false_negatives,sd_false_negatives,mean_suicidal_to_depressed,...,mean_suicidal_to_frustrated,sd_suicidal_to_frustrated,mean_suicidal_to_others,sd_suicidal_to_others,mean_corrected_false_negatives,sd_corrected_false_negatives,mean_introduced_false_negatives,sd_introduced_false_negatives,mean_unchanged_false_negatives,sd_unchanged_false_negatives
0,A0_flat,0.8652,0.0153,0.9154,0.0239,0.8892,0.0064,27.0,7.6158,15.2,...,4.3,2.4518,4.7,2.1108,0.0,0.0000,0.0,0.0000,27.0,7.6158
1,A1_conditional,0.8658,0.0151,0.9160,0.0234,0.8898,0.0064,26.8,7.4506,15.0,...,4.3,2.4518,4.7,2.1108,0.3,0.9487,0.1,0.3162,26.7,7.4095
2,A2_always_route,0.8694,0.0123,0.9157,0.0137,0.8918,0.0051,26.9,4.3576,15.2,...,4.4,2.5033,4.5,1.7795,1.5,3.1710,1.4,1.7127,25.5,5.2967


PASS: mechanism, BCR, and Suicidal analyses aggregated


## Reliability and selective prediction

ECE, Brier, NLL, macro ROC-AUC, and macro PR-AUC are valid only for the global
five-class probabilities. A1 and A2 reuse global maximum softmax solely as a
risk-ranking score; no CTPR probability vector or calibration metric is fabricated.

In [11]:
flat_reliability_seedwise = pd.DataFrame(reliability_rows)
flat_reliability_bins = pd.DataFrame(reliability_bin_rows)
risk_coverage = pd.DataFrame(risk_coverage_rows)
aurc_seedwise = pd.DataFrame(aurc_rows)

assert len(flat_reliability_seedwise) == len(SEEDS)
assert len(flat_reliability_bins) == len(SEEDS) * ECE_BINS
assert len(risk_coverage) == len(SEEDS) * 3 * len(validation)
assert len(aurc_seedwise) == len(SEEDS) * 3
assert set(aurc_seedwise["ranking_score"]) == {"global_max_softmax"}

flat_reliability_summary = pd.DataFrame(
    {
        "metric": [
            "ece", "brier", "nll", "macro_roc_auc", "macro_pr_auc"
        ],
        "mean": [
            flat_reliability_seedwise[column].mean()
            for column in (
                "ece", "brier", "nll", "macro_roc_auc", "macro_pr_auc"
            )
        ],
        "sample_sd": [
            flat_reliability_seedwise[column].std(ddof=1)
            for column in (
                "ece", "brier", "nll", "macro_roc_auc", "macro_pr_auc"
            )
        ],
    }
)
aurc_summary = summarize_frame(
    aurc_seedwise,
    ["system_id", "ranking_score"],
    ["aurc"],
)
forbidden_ctpr_probability_metrics = {
    "ctpr_ece", "ctpr_brier", "ctpr_nll",
    "ctpr_macro_roc_auc", "ctpr_macro_pr_auc",
}
exported_columns = set(flat_reliability_seedwise.columns)
exported_columns |= set(aurc_seedwise.columns)
assert forbidden_ctpr_probability_metrics.isdisjoint(exported_columns)

flat_reliability_seedwise.to_csv(
    OUTPUT_DIR / "phase11_flat_reliability_seedwise.csv",
    index=False,
)
flat_reliability_summary.to_csv(
    OUTPUT_DIR / "phase11_flat_reliability_summary.csv",
    index=False,
)
flat_reliability_bins.to_csv(
    OUTPUT_DIR / "phase11_flat_reliability_bins.csv",
    index=False,
)
risk_coverage.to_csv(
    OUTPUT_DIR / "phase11_risk_coverage.csv",
    index=False,
)
aurc_seedwise.to_csv(
    OUTPUT_DIR / "phase11_aurc_seedwise.csv",
    index=False,
)
aurc_summary.to_csv(
    OUTPUT_DIR / "phase11_aurc_summary.csv",
    index=False,
)
display(flat_reliability_summary.round(6))
display(aurc_summary.round(6))
print("PASS: valid Flat reliability and A0/A1/A2 risk-coverage complete")

,metric,mean,sample_sd
0,ece,0.093711,0.026297
1,brier,0.231654,0.016996
2,nll,0.589411,0.137968
3,macro_roc_auc,0.975872,0.002587
4,macro_pr_auc,0.922056,0.009381


,system_id,ranking_score,mean_aurc,sd_aurc
0,A0_flat,global_max_softmax,0.042530,0.005782
1,A1_conditional,global_max_softmax,0.042520,0.005782
2,A2_always_route,global_max_softmax,0.042488,0.005793


PASS: valid Flat reliability and A0/A1/A2 risk-coverage complete


## Lexical-cue robustness

Original, direct-mask, and cumulative operational-mask results are compared for Flat
and Conditional CTPR. Routing is recomputed from each masked representation. Changes
are descriptive shortcut-sensitivity evidence, not proof of semantic invariance.

In [12]:
lexical_seedwise = pd.DataFrame(lexical_metric_rows)
lexical_per_class = pd.DataFrame(lexical_per_class_rows)
routing_changes = pd.DataFrame(routing_change_rows)
assert len(lexical_seedwise) == len(SEEDS) * 3 * 2
assert len(lexical_per_class) == len(SEEDS) * 3 * 2 * len(LABELS)
assert len(routing_changes) == len(SEEDS) * 2

baseline_overall = lexical_seedwise[
    lexical_seedwise["variant"] == "original"
][
    [
        "model_seed", "system_id", "accuracy", "macro_f1",
        "balanced_accuracy", "routing_coverage",
    ]
].rename(
    columns={
        "accuracy": "original_accuracy",
        "macro_f1": "original_macro_f1",
        "balanced_accuracy": "original_balanced_accuracy",
        "routing_coverage": "original_routing_coverage",
    }
)
lexical_seedwise = lexical_seedwise.merge(
    baseline_overall,
    on=["model_seed", "system_id"],
    how="left",
)
for metric in (
    "accuracy", "macro_f1", "balanced_accuracy", "routing_coverage"
):
    lexical_seedwise[f"delta_{metric}_vs_original"] = (
        lexical_seedwise[metric]
        - lexical_seedwise[f"original_{metric}"]
    )

baseline_class = lexical_per_class[
    lexical_per_class["variant"] == "original"
][
    ["model_seed", "system_id", "class_label", "precision", "recall", "f1"]
].rename(
    columns={
        "precision": "original_precision",
        "recall": "original_recall",
        "f1": "original_f1",
    }
)
lexical_per_class = lexical_per_class.merge(
    baseline_class,
    on=["model_seed", "system_id", "class_label"],
    how="left",
)
for metric in ("precision", "recall", "f1"):
    lexical_per_class[f"delta_{metric}_vs_original"] = (
        lexical_per_class[metric]
        - lexical_per_class[f"original_{metric}"]
    )

lexical_summary = summarize_frame(
    lexical_seedwise,
    ["system_id", "variant"],
    [
        "accuracy", "macro_f1", "balanced_accuracy",
        "routing_coverage", "delta_accuracy_vs_original",
        "delta_macro_f1_vs_original",
        "delta_balanced_accuracy_vs_original",
        "delta_routing_coverage_vs_original",
    ],
)
lexical_seedwise.to_csv(
    OUTPUT_DIR / "phase11_lexical_robustness_seedwise.csv",
    index=False,
)
lexical_summary.to_csv(
    OUTPUT_DIR / "phase11_lexical_robustness_summary.csv",
    index=False,
)
lexical_per_class.to_csv(
    OUTPUT_DIR / "phase11_lexical_per_class.csv",
    index=False,
)
routing_changes.to_csv(
    OUTPUT_DIR / "phase11_routing_changes.csv",
    index=False,
)
display(lexical_summary.round(6))
display(routing_changes.groupby("variant").mean(numeric_only=True).round(3))
print("PASS: lexical robustness and routing-change analyses complete")

,system_id,variant,mean_accuracy,sd_accuracy,mean_macro_f1,sd_macro_f1,mean_balanced_accuracy,sd_balanced_accuracy,mean_routing_coverage,sd_routing_coverage,mean_delta_accuracy_vs_original,sd_delta_accuracy_vs_original,mean_delta_macro_f1_vs_original,sd_delta_macro_f1_vs_original,mean_delta_balanced_accuracy_vs_original,sd_delta_balanced_accuracy_vs_original,mean_delta_routing_coverage_vs_original,sd_delta_routing_coverage_vs_original
0,A0_flat,direct_mask,0.831275,0.005181,0.825913,0.005267,0.821849,0.006597,0.000000,0.000000,-0.035302,0.005723,-0.037285,0.006354,-0.040374,0.007038,0.000000,0.000000
1,A0_flat,operational_mask,0.795839,0.008291,0.791260,0.008016,0.785739,0.009261,0.000000,0.000000,-0.070738,0.008461,-0.071937,0.008774,-0.076483,0.009719,0.000000,0.000000
2,A0_flat,original,0.866577,0.003315,0.863198,0.003567,0.862223,0.003770,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,A1_conditional,direct_mask,0.832282,0.005279,0.826991,0.005153,0.822928,0.006226,0.007450,0.004782,-0.035235,0.005582,-0.037147,0.006093,-0.040225,0.006771,0.000872,0.001186
4,A1_conditional,operational_mask,0.796779,0.008261,0.792245,0.008000,0.786794,0.009028,0.008926,0.006360,-0.070738,0.008437,-0.071893,0.008746,-0.076359,0.009556,0.002349,0.002175
5,A1_conditional,original,0.867517,0.004066,0.864138,0.004309,0.863153,0.004470,0.006577,0.004347,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,model_seed,original_routed_n,masked_routed_n,route_entries,route_exits,routed_pair_changes,routing_indicator_changes
variant,,,,,,,
direct_mask,777.5,9.8,11.1,2.4,1.1,0.0,3.5
operational_mask,777.5,9.8,13.3,6.0,2.5,0.0,8.5


PASS: lexical robustness and routing-change analyses complete


## Figures

Reviewer-facing plots use equal-seed means and sample-standard-deviation error bars.
Both PNG and SVG are saved. No inferential significance markers are added.

In [13]:
def save_figure(figure, stem):
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / f"{stem}.png", dpi=180, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{stem}.svg", bbox_inches="tight")
    plt.close(figure)


def grouped_bar(frame, category, values, title, ylabel, stem):
    systems = ["A1_conditional", "A2_always_route"]
    value_names = list(values)
    categories = list(dict.fromkeys(frame[category].tolist()))
    x = np.arange(len(categories), dtype=float)
    width = 0.8 / (len(systems) * len(value_names))
    figure, axis = plt.subplots(figsize=(max(8, len(categories) * 1.4), 5))
    offset_index = 0
    for system_id in systems:
        for value in value_names:
            means = []
            errors = []
            for name in categories:
                selected = frame[
                    (frame["system_id"] == system_id)
                    & (frame[category] == name)
                ][value]
                means.append(selected.mean())
                errors.append(selected.std(ddof=1))
            offset = (offset_index - 2.5) * width
            axis.bar(
                x + offset,
                means,
                width,
                yerr=errors,
                capsize=3,
                label=f"{system_id}: {value}",
            )
            offset_index += 1
    axis.axhline(0, color="black", linewidth=0.8)
    axis.set_xticks(x)
    axis.set_xticklabels(categories, rotation=25, ha="right")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.legend(fontsize=8)
    save_figure(figure, stem)


grouped_bar(
    correction_by_pair,
    "pair",
    ("correction", "harm", "net_corrections"),
    "Corrections, harms, and net corrections by hard pair",
    "Mean rows across seeds",
    "correction_harm_net_by_pair",
)
grouped_bar(
    correction_by_class,
    "true_class",
    ("correction", "harm", "net_corrections"),
    "Corrections, harms, and net corrections by true class",
    "Mean rows across seeds",
    "correction_harm_net_by_true_class",
)

for value, label, stem in (
    ("confidence_mean", "Seed-mean global probability", "confidence_corrected_harmed"),
    ("margin_mean", "Seed-mean top-two margin", "margin_corrected_harmed"),
):
    selected = confidence_margin_summary[
        confidence_margin_summary["category"].isin(["correction", "harm"])
    ]
    groups = []
    names = []
    for system_id in ("A1_conditional", "A2_always_route"):
        for category in ("correction", "harm"):
            values = selected[
                (selected["system_id"] == system_id)
                & (selected["category"] == category)
            ][value].dropna().to_numpy()
            if len(values):
                groups.append(values)
                names.append(system_id.replace("_", " ") + "\n" + category)
    figure, axis = plt.subplots(figsize=(8, 5))
    if groups:
        axis.boxplot(groups, tick_labels=names, showfliers=False)
    axis.set_ylabel(label)
    axis.set_title(label + ": corrected versus harmed cases")
    save_figure(figure, stem)

hard_bcr = bcr_seedwise[bcr_seedwise["pair"].isin(HARD_PAIRS)]
figure, axis = plt.subplots(figsize=(10, 5))
x = np.arange(len(HARD_PAIRS), dtype=float)
systems = ["A0_flat", "A1_conditional", "A2_always_route"]
for index, system_id in enumerate(systems):
    means = []
    errors = []
    for pair_name in HARD_PAIRS:
        values = hard_bcr[
            (hard_bcr["system_id"] == system_id)
            & (hard_bcr["pair"] == pair_name)
        ]["bcr"]
        means.append(values.mean())
        errors.append(values.std(ddof=1))
    axis.bar(
        x + (index - 1) * 0.25,
        means,
        0.25,
        yerr=errors,
        capsize=3,
        label=system_id,
    )
axis.set_xticks(x)
axis.set_xticklabels(HARD_PAIRS, rotation=20, ha="right")
axis.set_ylabel("Boundary confusion rate")
axis.set_title("Hard-pair BCR before and after routing")
axis.legend()
save_figure(figure, "hard_pair_bcr_before_after")

destination_columns = [
    "suicidal_to_depressed", "suicidal_to_anxious",
    "suicidal_to_frustrated", "suicidal_to_others",
]
figure, axis = plt.subplots(figsize=(9, 5))
x = np.arange(len(destination_columns), dtype=float)
for index, system_id in enumerate(systems):
    selected = suicidal_seedwise[suicidal_seedwise["system_id"] == system_id]
    means = [selected[column].mean() for column in destination_columns]
    errors = [selected[column].std(ddof=1) for column in destination_columns]
    axis.bar(
        x + (index - 1) * 0.25,
        means,
        0.25,
        yerr=errors,
        capsize=3,
        label=system_id,
    )
axis.set_xticks(x)
axis.set_xticklabels(
    [column.replace("suicidal_to_", "") for column in destination_columns]
)
axis.set_ylabel("Mean false-negative rows across seeds")
axis.set_title("Suicidal false-negative destinations")
axis.legend()
save_figure(figure, "suicidal_false_negative_destinations")

bin_summary = flat_reliability_bins.groupby("bin_id", as_index=False).agg(
    mean_confidence=("mean_confidence", "mean"),
    mean_accuracy=("empirical_accuracy", "mean"),
    sd_accuracy=("empirical_accuracy", "std"),
)
figure, axis = plt.subplots(figsize=(6, 6))
axis.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect")
valid = bin_summary["mean_confidence"].notna()
axis.errorbar(
    bin_summary.loc[valid, "mean_confidence"],
    bin_summary.loc[valid, "mean_accuracy"],
    yerr=bin_summary.loc[valid, "sd_accuracy"].fillna(0),
    marker="o",
    capsize=3,
    label="Flat mean ± seed SD",
)
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.set_xlabel("Mean confidence")
axis.set_ylabel("Empirical accuracy")
axis.set_title("Flat MentalRoBERTa reliability")
axis.legend()
save_figure(figure, "flat_reliability_diagram")

curve_summary = risk_coverage.groupby(
    ["system_id", "retained_n"],
    as_index=False,
).agg(mean_coverage=("coverage", "mean"), mean_risk=("risk", "mean"))
figure, axis = plt.subplots(figsize=(7, 5))
for system_id in systems:
    selected = curve_summary[curve_summary["system_id"] == system_id]
    axis.plot(selected["mean_coverage"], selected["mean_risk"], label=system_id)
axis.set_xlabel("Coverage")
axis.set_ylabel("Risk (1 - accuracy)")
axis.set_title("Risk-coverage using frozen global confidence ranking")
axis.legend()
save_figure(figure, "risk_coverage")

figure, axis = plt.subplots(figsize=(9, 5))
variants = ["original", "direct_mask", "operational_mask"]
x = np.arange(len(variants), dtype=float)
for index, system_id in enumerate(("A0_flat", "A1_conditional")):
    means = []
    errors = []
    for variant in variants:
        selected = lexical_seedwise[
            (lexical_seedwise["system_id"] == system_id)
            & (lexical_seedwise["variant"] == variant)
        ]["macro_f1"]
        means.append(selected.mean())
        errors.append(selected.std(ddof=1))
    axis.bar(
        x + (index - 0.5) * 0.35,
        means,
        0.35,
        yerr=errors,
        capsize=3,
        label=system_id,
    )
axis.set_xticks(x)
axis.set_xticklabels(variants)
axis.set_ylabel("Macro-F1")
axis.set_title("Lexical-cue robustness")
axis.legend()
save_figure(figure, "lexical_macro_f1")

masked_class = lexical_per_class[
    lexical_per_class["variant"] != "original"
]
class_delta = masked_class.groupby(
    ["system_id", "variant", "class_label"],
    as_index=False,
)["delta_f1_vs_original"].mean()
figure, axis = plt.subplots(figsize=(11, 5))
labels_for_plot = []
values_for_plot = []
for row in class_delta.itertuples():
    labels_for_plot.append(
        f"{row.system_id}\n{row.variant}\n{row.class_label}"
    )
    values_for_plot.append(row.delta_f1_vs_original)
axis.bar(np.arange(len(values_for_plot)), values_for_plot)
axis.axhline(0, color="black", linewidth=0.8)
axis.set_xticks(np.arange(len(values_for_plot)))
axis.set_xticklabels(labels_for_plot, rotation=70, ha="right", fontsize=7)
axis.set_ylabel("Mean F1 change from original")
axis.set_title("Per-class lexical-mask sensitivity")
save_figure(figure, "lexical_per_class_f1_change")

conditional_coverage = lexical_seedwise[
    lexical_seedwise["system_id"] == "A1_conditional"
]
figure, axis = plt.subplots(figsize=(8, 5))
means = [
    conditional_coverage[
        conditional_coverage["variant"] == variant
    ]["routing_coverage"].mean()
    for variant in variants
]
errors = [
    conditional_coverage[
        conditional_coverage["variant"] == variant
    ]["routing_coverage"].std(ddof=1)
    for variant in variants
]
axis.bar(variants, means, yerr=errors, capsize=3)
axis.set_ylabel("Routing coverage")
axis.set_title("Conditional routing coverage under lexical masks")
save_figure(figure, "lexical_routing_coverage")

expected_stems = {
    "correction_harm_net_by_pair", "correction_harm_net_by_true_class",
    "confidence_corrected_harmed", "margin_corrected_harmed",
    "hard_pair_bcr_before_after", "suicidal_false_negative_destinations",
    "flat_reliability_diagram", "risk_coverage", "lexical_macro_f1",
    "lexical_per_class_f1_change", "lexical_routing_coverage",
}
for stem in expected_stems:
    assert (FIGURE_DIR / f"{stem}.png").is_file()
    assert (FIGURE_DIR / f"{stem}.svg").is_file()
print("PASS: reviewer-facing PNG and SVG figures saved")

PASS: reviewer-facing PNG and SVG figures saved


## Export and integrity verification

The final gate writes the protocol and execution summary, verifies every expected
scientific table, rejects forbidden input access, builds an exact SHA-256 manifest,
and creates a deterministic ZIP with a fixed timestamp.

In [14]:
phase10_reproduction.to_csv(
    OUTPUT_DIR / "phase11_phase10_reproduction.csv",
    index=False,
)
pd.DataFrame(ACCESSED_INPUTS).to_csv(
    OUTPUT_DIR / "phase11_input_inventory.csv",
    index=False,
)

protocol = {
    "phase": 11,
    "title": "Mechanism, Reliability, and Lexical Robustness",
    "labels": list(LABELS),
    "model_seeds": list(SEEDS),
    "H": list(HARD_PAIRS),
    "K": K_FIXED,
    "frozen_tau": FROZEN_TAU,
    "always_route_tau": ALWAYS_ROUTE_TAU,
    "high_confidence_threshold": HIGH_CONFIDENCE_THRESHOLD,
    "ece_bins": ECE_BINS,
    "ece_bin_rule": "min(floor(confidence * 15), 14)",
    "nll_epsilon": NLL_EPSILON,
    "risk_ranking_score": "global_max_softmax",
    "aurc_rule": "mean empirical risk over retained_n 1..1490",
    "ctpr_probability_metrics_computed": False,
    "lexical_mask_version": MASK_SPEC["version"],
    "validation_sha256": VALIDATION_SHA256,
    "phase10_zip_sha256": PHASE10_ZIP_SHA256,
    "test_accessed": False,
}
write_json(OUTPUT_DIR / "phase11_protocol.json", protocol)

required_outputs = (
    "phase11_protocol.json",
    "phase11_input_inventory.csv",
    "phase11_phase10_reproduction.csv",
    "phase11_lexical_mask_spec.json",
    "phase11_lexical_mask_coverage.csv",
    "phase11_original_seedwise_metrics.csv",
    "phase11_original_metric_summary.csv",
    "phase11_correction_harm_seedwise.csv",
    "phase11_correction_harm_by_pair.csv",
    "phase11_correction_harm_by_class.csv",
    "phase11_confidence_margin_summary.csv",
    "phase11_high_confidence_errors.csv",
    "phase11_routed_subset_metrics.csv",
    "phase11_bcr_seedwise.csv",
    "phase11_bcr_summary.csv",
    "phase11_suicidal_seedwise.csv",
    "phase11_suicidal_summary.csv",
    "phase11_flat_reliability_seedwise.csv",
    "phase11_flat_reliability_bins.csv",
    "phase11_risk_coverage.csv",
    "phase11_aurc_seedwise.csv",
    "phase11_aurc_summary.csv",
    "phase11_lexical_robustness_seedwise.csv",
    "phase11_lexical_robustness_summary.csv",
    "phase11_lexical_per_class.csv",
    "phase11_routing_changes.csv",
)
missing_outputs = [
    name for name in required_outputs if not (OUTPUT_DIR / name).is_file()
]
assert not missing_outputs, missing_outputs
assert len(list(PREDICTION_DIR.glob("*.csv"))) == len(SEEDS) * 3
assert phase10_reproduction["status"].eq("PASS").sum() == len(SEEDS)
for frame, columns in (
    (
        original_seedwise,
        ("accuracy", "macro_f1", "balanced_accuracy"),
    ),
    (
        flat_reliability_seedwise,
        ("ece", "brier", "nll", "macro_roc_auc", "macro_pr_auc"),
    ),
    (aurc_seedwise, ("aurc",)),
    (
        lexical_seedwise,
        (
            "accuracy", "macro_f1", "balanced_accuracy",
            "routing_coverage",
        ),
    ),
):
    assert np.isfinite(frame[list(columns)].to_numpy(dtype=float)).all()
assert (correction_seedwise["routed_n"] >= 0).all()
assert set(lexical_seedwise["variant"]) == {
    "original", "direct_mask", "operational_mask"
}
assert lexical_seedwise.groupby(
    ["system_id", "variant"]
)["model_seed"].nunique().eq(len(SEEDS)).all()
assert routed_subset_metrics.loc[
    ~routed_subset_metrics["defined"], "missing_reason"
].eq("empty_subset").all()

input_inventory = pd.DataFrame(ACCESSED_INPUTS)
forbidden_access = input_inventory[
    input_inventory["path"].map(
        lambda value: any(
            marker in Path(value).name.lower()
            for marker in ("_test", "combined", "train_locked")
        )
    )
]
assert forbidden_access.empty

execution_summary = {
    "phase": 11,
    "status": "PASS",
    "seed_count": len(SEEDS),
    "phase10_reproduction_pass_count": int(
        phase10_reproduction["status"].eq("PASS").sum()
    ),
    "validation_rows": len(validation),
    "verified_phase8_heads": verified_head_count,
    "original_system_seed_rows": len(original_seedwise),
    "lexical_system_seed_rows": len(lexical_seedwise),
    "risk_coverage_rows": len(risk_coverage),
    "frozen_tau": FROZEN_TAU,
    "test_accessed": False,
}
write_json(
    OUTPUT_DIR / "phase11_execution_summary.json",
    execution_summary,
)

report_lines = [
    "# Phase 11 Mechanism, Reliability, and Lexical Robustness Report",
    "",
    "Status: PASS",
    "",
    f"Frozen H: {list(HARD_PAIRS)}",
    f"Frozen K: {K_FIXED}",
    f"Frozen tau: {FROZEN_TAU:.2f}",
    f"Phase 10 reproductions: {len(phase10_reproduction)}/10 PASS",
    "Test data accessed: False",
    "",
    "## Original-system summary",
    "",
    "```csv",
    original_summary.to_csv(index=False).strip(),
    "```",
    "",
    "## Correction and harm summary",
    "",
    "```csv",
    correction_summary.to_csv(index=False).strip(),
    "```",
    "",
    "## Flat reliability summary",
    "",
    "```csv",
    flat_reliability_summary.to_csv(index=False).strip(),
    "```",
    "",
    "## Lexical robustness summary",
    "",
    "```csv",
    lexical_summary.to_csv(index=False).strip(),
    "```",
    "",
    "All results are descriptive development-validation analyses. Lexical masking",
    "does not prove semantic understanding or deployment safety.",
]
(OUTPUT_DIR / "phase11_report.md").write_text(
    "\n".join(report_lines) + "\n"
)

manifest_rows = []
for result_path in sorted(
    path for path in OUTPUT_DIR.rglob("*") if path.is_file()
):
    manifest_rows.append(
        {
            "path": result_path.relative_to(OUTPUT_DIR).as_posix(),
            "sha256": sha256_file(result_path),
            "bytes": result_path.stat().st_size,
        }
    )
pd.DataFrame(manifest_rows).to_csv(
    OUTPUT_DIR / "sha256_manifest.csv",
    index=False,
)
verify_manifest(OUTPUT_DIR)

zip_path = (
    WORKING_ROOT
    / "MentalDistress_Phase11_Mechanism_Reliability_Robustness_Results.zip"
)
zip_sha256 = write_deterministic_zip(OUTPUT_DIR, zip_path)

print("=" * 76)
print("PHASE 11 COMPLETE")
print("Frozen tau:", FROZEN_TAU)
print("Phase 10 reproductions:", len(phase10_reproduction), "/ 10 PASS")
print("Results ZIP:", zip_path)
print("ZIP SHA256:", zip_sha256)
print("Test data accessed: False")
print("=" * 76)

PHASE 11 COMPLETE
Frozen tau: 0.1
Phase 10 reproductions: 10 / 10 PASS
Results ZIP: /kaggle/working/MentalDistress_Phase11_Mechanism_Reliability_Robustness_Results.zip
ZIP SHA256: f3a0ae1499d8cee01e370cadcf3b09ee3ea36bbd9f0eb3e0bd0428fe9abc1068
Test data accessed: False
